# STG_7.2 — Optimización de hiperparámetros del modelo ganador (spec 013)

Toma el modelo/escenario ganador de STG_6.2 — **LGBMClassifier, Escenario A (PRO fuera de
LLA)** — y busca mejores hiperparámetros con `RandomizedSearchCV`, usando el **mismo**
`TimeSeriesSplit` de STG_6.2.

Regla obligatoria: la búsqueda usa `cv=TimeSeriesSplit`, nunca CV aleatoria.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, cross_val_score
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier

DATA_DIR  = Path("../data")
SPEC_DIR  = Path("../specs/013-reentrenar-modelo-features-autor")
RANDOM_STATE = 42


## T17a — Carga y reconstrucción del split del ganador (Escenario A)

In [2]:
# Misma carga y split que STG_6.2 (reproducibilidad garantizada)
df = pd.read_csv(DATA_DIR / "df_entrenamiento_autor.csv", parse_dates=["fecha_votacion"])
df = df.sort_values("fecha_votacion").reset_index(drop=True)

META   = ["diputado", "titulo_base", "fecha_votacion", "id_votacion", "bloque", "provincia", "tema_label"]
TARGET = "voto"
FEATURES_COMUNES = [c for c in df.columns if c not in META + [TARGET]
                     and c not in ("es_oficialista", "coincide_bloque_autor",
                                    "es_oficialista_b", "coincide_bloque_autor_b")]
# Escenario A (ganador): PRO fuera de la coalición de Milei
FEATURES = FEATURES_COMUNES + ["es_oficialista", "coincide_bloque_autor"]

le_voto = LabelEncoder()
df["voto_enc"] = le_voto.fit_transform(df[TARGET])

corte_idx = int(len(df) * 0.80)
df_train  = df.iloc[:corte_idx].copy()
df_hold   = df.iloc[corte_idx:].copy()
X_train   = df_train[FEATURES].values
y_train   = df_train["voto_enc"].values
X_hold    = df_hold[FEATURES].values
y_hold    = df_hold["voto_enc"].values

tscv = TimeSeriesSplit(n_splits=5)

fecha_corte = df.iloc[corte_idx]["fecha_votacion"]
print(f"Train  : {len(df_train):,} filas hasta {df_train['fecha_votacion'].max().date()}")
print(f"Holdout: {len(df_hold):,} filas desde {df_hold['fecha_votacion'].min().date()}")
print(f"Features: {len(FEATURES)} (Escenario A)")

assert df_train["fecha_votacion"].max() <= df_hold["fecha_votacion"].min()
assert len(df) == 20608
print()
print("✓ T17a PASA: split idéntico a STG_6.2 (Escenario A) verificado.")


Train  : 16,486 filas hasta 2025-12-18
Holdout: 4,122 filas desde 2025-12-18
Features: 398 (Escenario A)

✓ T17a PASA: split idéntico a STG_6.2 (Escenario A) verificado.


## T17b — Búsqueda de hiperparámetros con RandomizedSearchCV

In [3]:
import scipy.stats as stats

modelo_base = LGBMClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

param_dist = {
    'n_estimators'     : [100, 200, 300, 400, 500, 600],
    'learning_rate'    : [0.01, 0.03, 0.05, 0.08, 0.10, 0.15],
    'num_leaves'       : [31, 47, 63, 80, 100, 127],
    'min_child_samples': [10, 20, 30, 50, 80],
    'colsample_bytree' : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
}

N_ITER = 15  # reducido de 30 (spec 009) a 15 por restricción de tiempo de cómputo; ver DECISIONES.md

buscador = RandomizedSearchCV(
    estimator   = modelo_base,
    param_distributions = param_dist,
    n_iter      = N_ITER,
    cv          = tscv,
    scoring     = 'f1_macro',
    random_state= RANDOM_STATE,
    n_jobs      = 1,       # evitar doble paralelismo con LightGBM (bug documentado spec 009)
    verbose     = 1,
    refit       = True,
)

print(f"Iniciando búsqueda de hiperparámetros ({N_ITER} iteraciones x 5 folds = {N_ITER*5} ajustes)...")
buscador.fit(X_train, y_train)

print()
print("Mejores hiperparámetros encontrados:")
for k, v in buscador.best_params_.items():
    print(f"  {k:<22}: {v}")
print(f"  CV F1-macro (mejor)  : {buscador.best_score_:.4f}")
print()
print("✓ T17b PASA.")


Iniciando búsqueda de hiperparámetros (15 iteraciones x 5 folds = 75 ajustes)...
Fitting 5 folds for each of 15 candidates, totalling 75 fits



Mejores hiperparámetros encontrados:
  subsample             : 0.6
  num_leaves            : 100
  n_estimators          : 500
  min_child_samples     : 80
  learning_rate         : 0.01
  colsample_bytree      : 0.5
  CV F1-macro (mejor)  : 0.4682

✓ T17b PASA.


## T17c — Comparar modelo por defecto vs. afinado

In [4]:
modelo_afinado = buscador.best_estimator_

# Modelo por defecto de STG_6.2 (el que ganó la comparación de 12)
modelo_default = LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=63,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)

cv_default = cross_val_score(modelo_default, X_train, y_train,
                             cv=tscv, scoring='f1_macro', n_jobs=1).mean()

modelo_default.fit(X_train, y_train)
hold_default = f1_score(y_hold, modelo_default.predict(X_hold), average='macro')
hold_afinado = f1_score(y_hold, modelo_afinado.predict(X_hold), average='macro')

comparacion = pd.DataFrame([
    {'Modelo': 'LightGBM por defecto (STG_6.2, Esc. A)',
     'CV F1-macro': round(cv_default, 4),
     'Holdout F1-macro': round(hold_default, 4)},
    {'Modelo': 'LightGBM afinado (STG_7.2, Esc. A)',
     'CV F1-macro': round(buscador.best_score_, 4),
     'Holdout F1-macro': round(hold_afinado, 4)},
])
print(comparacion.to_string(index=False))
print()
mejora_hold = (hold_afinado - hold_default) / hold_default * 100
print(f"Mejora en holdout: {mejora_hold:+.1f}%")

comparacion.to_csv(SPEC_DIR / 'comparacion_default_vs_afinado_autor.csv', index=False)
print("Guardado: comparacion_default_vs_afinado_autor.csv")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
x = ['Por defecto\n(STG_6.2)', 'Afinado\n(STG_7.2)']
cv_vals   = [cv_default, buscador.best_score_]
hold_vals = [hold_default, hold_afinado]
xi = range(len(x))
ancho = 0.35
b1 = ax.bar([i - ancho/2 for i in xi], cv_vals,   ancho, label='CV F1-macro',      color='#5B9BD5')
b2 = ax.bar([i + ancho/2 for i in xi], hold_vals,  ancho, label='Holdout F1-macro', color='#ED7D31')
ax.set_xticks(list(xi)); ax.set_xticklabels(x)
ax.set_ylim(0, 0.7); ax.set_ylabel('F1-macro')
ax.set_title('LightGBM (Escenario A): por defecto vs. afinado', fontsize=12, fontweight='bold')
ax.legend()
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.008,
            f'{b.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(SPEC_DIR / 'comparacion_tuning_autor.png', dpi=150, bbox_inches='tight')
plt.close()
print("Guardado: comparacion_tuning_autor.png")
print()
print("✓ T17c PASA.")


                                Modelo  CV F1-macro  Holdout F1-macro
LightGBM por defecto (STG_6.2, Esc. A)       0.4061            0.5809
    LightGBM afinado (STG_7.2, Esc. A)       0.4682            0.5341

Mejora en holdout: -8.1%
Guardado: comparacion_default_vs_afinado_autor.csv


Guardado: comparacion_tuning_autor.png

✓ T17c PASA.
